# SETU-RAG on Google Colab — setup + run
Code-switch-aware multilingual **speech-to-speech** RAG (22 Indian languages).

**First:** Runtime → Change runtime type → **T4 GPU**. Internet is on by default. See `COLAB.md`.

## 1 · Check the GPU

In [ ]:
!nvidia-smi -L

## 2 · Get the code in
Upload `setu-rag.zip` via the Files panel (or mount Drive / git clone — see COLAB.md).

In [ ]:
import os, zipfile
if os.path.exists('setu-rag.zip'):
    zipfile.ZipFile('setu-rag.zip').extractall('.')
if os.path.isdir('setu-rag'):
    os.chdir('setu-rag')
print('cwd:', os.getcwd()); print(sorted(os.listdir('.'))[:12])

## 3 · (Optional) Hugging Face token via Colab Secrets (🔑 panel → HF_TOKEN)

In [ ]:
try:
    from google.colab import userdata
    import os
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login; login(os.environ['HF_TOKEN']); print('HF login OK')
except Exception as e:
    print('Skipping HF login (add HF_TOKEN secret to enable model downloads):', e)

## 4 · Install dependencies

In [ ]:
!pip -q install -r requirements.txt
# !pip -q install git+https://github.com/huggingface/parler-tts.git   # TTS
# IndicConformer (NeMo) / IndicTrans2 / IndicLID: see their GitHub READMEs

## 5 · Smoke test — implemented novel logic (no downloads)

In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from setu_rag.front_end.language_id import LanguageIdentifier
from setu_rag.front_end.cmi import compute_cmi
from setu_rag.router.adaptive_router import decide_route
lid = LanguageIdentifier().load()
for q in ['mera refund kab tak aayega order cancel kiya tha',
          'why is my coupon not applying to the cart']:
    t = lid.tag(q); c = compute_cmi(t); fr = sum(x.script=='Latn' for x in t)/len(t)
    print(f'CMI={c.cmi:.2f} matrix={c.matrix_lang} -> {decide_route(c, fr).route.value}  | {q[:40]}')

## 6 · Speech-layer logic

In [ ]:
from setu_rag.speech.lid_fusion import fuse
from setu_rag.speech.tts import style_description
toks = lid.tag('mera refund kab tak aayega order cancel kiya')
print(fuse({'hin':0.8,'eng':0.2}, toks, alpha=0.4))
print(style_description('hin_Deva', 0.37))

## 7 · Full pipeline / voice demo (after wiring `# TODO` model stubs)

In [ ]:
# !python scripts/build_index.py --faqs data/faqs.sample.jsonl --out index
# in scripts/serve_voice.py: demo.launch(share=True)
# !python scripts/serve_voice.py